# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
%run ./utils/logger

In [0]:
run_id = get_run_id()
print(run_id)

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
orders_df = spark.sql(f"""
SELECT *, (_metadata.file_name) as file_name
FROM read_files(
    'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/inbound/orders*.csv',
    format => 'csv',
    inferSchema => true
)
""")

In [0]:
display(orders_df)

In [0]:
files_recieved = orders_df.count()

if files_recieved > 0:
    print(f'Number of files recieved :{files_recieved}')
else:
    print('No files present in adls path')

In [0]:
try:
    spark.sql(f""" create table if not exists {catalog}.{schema}.orders_stage
              as
              SELECT *, (_metadata.file_name) as file_name
              FROM read_files(
                  'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/inbound/orders*.csv',format => 'csv',
                  inferSchema => true
                  )
                  """)

    log_run(run_id, "orders_ingest_pipeline", "orders_raw", "SUCCESS", "orders raw data satge load completed")

except Exception as e:
    log_run(run_id, "orders_ingest_pipeline", "orders_raw", "FAILED", error_message=str(e))
    raise 

In [0]:
%sql
describe table extended orders_stage

In [0]:
%sql
describe table extended orders

In [0]:
try:

    spark.sql(f"""
        TRUNCATE TABLE {catalog}.bronze.orders
    """)

    spark.sql(f"""
        INSERT INTO {catalog}.bronze.orders
        SELECT
            CAST(order_id AS STRING)         AS order_id,
            CAST(customer_id AS STRING)      AS customer_id,
            CAST(order_date AS DATE)         AS order_date,
            TRIM(order_status)               AS order_status,
            TRIM(payment_method)             AS payment_method,
            TRIM(order_channel)              AS order_channel,
            CURRENT_TIMESTAMP()              AS ingestion_time,
            file_name                        AS source_file
        FROM {catalog}.bronze.orders_stage
    """)

    log_run(
        run_id,
        "orders_ingest_pipeline",
        "bronze_load",
        "SUCCESS",
        "Orders bronze load completed"
    )

except Exception as e:

    log_run(
        run_id,
        "orders_ingest_pipeline",
        "bronze_load",
        "FAILED",
        error_message=str(e)
    )

    raise

In [0]:
%sql
DROP TABLE commerce_raw_dev.bronze.orders_stage;